# AI-Driven Audit & Compliance Validator
## TCS AMD Hackathon 2026

This notebook demonstrates the full audit pipeline:
1. **Document Intake** — PDF parsing with PyMuPDF + pdfplumber (OCR optional)
2. **Two-RAG System** — ChromaDB with bge-large-en-v1.5 embeddings
3. **103 Built-in Rules** — Comprehensive SEBI ICDR/LODR/SAST compliance rules
4. **5-Layer Validation** — Deterministic + NLP + Semantic + Numerical + Cross-Reference
5. **Self-Reflection Critic** — Reviews findings, catches false positives
6. **Confidence Scoring** — Multi-signal weighted scores
7. **Audit Report** — PDF export with full audit trail

---

## Step 0: Install Dependencies
Run this cell once to install all required packages.

In [ ]:
# Install dependencies (run once)
!pip install -q openai sentence-transformers chromadb \
    langgraph langchain-core langchain-openai \
    PyMuPDF pdfplumber Pillow \
    fpdf2 pandas scikit-learn tqdm

## Step 1: Configuration

**IMPORTANT:** Adjust the paths below to match your environment:
- `AUDIT_BASE_DIR` — Directory containing `compliance_guidelines/` and `ipo_documents/`
- `AUDIT_SHARED_DIR` — Persistent storage directory (survives notebook restart)
- `VLLM_BASE_URL` — Your vLLM server endpoint

In [ ]:
import os
import sys

# ============================================================
# CONFIGURE THESE PATHS FOR YOUR ENVIRONMENT
# ============================================================

# Option A: If running on AMD Cloud (notebook.amd.com)
# os.environ['AUDIT_BASE_DIR'] = '/home/user/dataset'        # Where you uploaded the PDFs
# os.environ['AUDIT_SHARED_DIR'] = '/home/user/shared'       # Persistent shared folder
# os.environ['VLLM_BASE_URL'] = 'http://localhost:8000/v1'   # vLLM endpoint

# Option B: If running locally (Windows)
os.environ['AUDIT_BASE_DIR'] = r'E:\Dataset for TCS AI Hackathon'
os.environ['AUDIT_SHARED_DIR'] = r'E:\Dataset for TCS AI Hackathon\shared'
os.environ['VLLM_BASE_URL'] = 'http://localhost:8000/v1'

# OCR is disabled by default (PDFs have selectable text)
# Set to 'true' only if you have scanned image PDFs
os.environ['USE_OCR'] = 'false'

# Add project root to Python path
project_root = os.environ['AUDIT_BASE_DIR']
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f'Project root: {project_root}')
print(f'Shared dir:   {os.environ["AUDIT_SHARED_DIR"]}')
print(f'vLLM URL:     {os.environ["VLLM_BASE_URL"]}')
print(f'OCR enabled:  {os.environ["USE_OCR"]}')

In [ ]:
# Verify configuration and directory structure
from src.config import ensure_dirs, print_config, COMPLIANCE_DIR, IPO_DIR

ensure_dirs()
print_config()

# Verify data exists
compliance_files = os.listdir(COMPLIANCE_DIR) if os.path.isdir(COMPLIANCE_DIR) else []
ipo_files = [f for f in os.listdir(IPO_DIR) if f.endswith('.pdf')] if os.path.isdir(IPO_DIR) else []

print(f'\nCompliance PDFs: {len(compliance_files)} files')
for f in compliance_files:
    print(f'  - {f}')

print(f'\nIPO Documents: {len(ipo_files)} PDFs')
print(f'  First 5: {ipo_files[:5]}')

## Step 2: Start vLLM Server

**Run this in a separate terminal** (not in this notebook):

```bash
python -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen2.5-72B-Instruct \
    --dtype bfloat16 \
    --tensor-parallel-size 1 \
    --port 8000 \
    --max-model-len 16384 \
    --gpu-memory-utilization 0.90 \
    --trust-remote-code
```

Wait for the server to show `Uvicorn running on http://0.0.0.0:8000` before proceeding.

In [ ]:
# Test LLM connection
from src.llm_client import LLMClient

llm = LLMClient()

# Health check
print('Testing LLM connection...')
is_healthy = llm.health_check()
if is_healthy:
    print('LLM server is running and responsive!')
else:
    print('ERROR: LLM server is not responding.')
    print('Make sure vLLM is running on the correct port.')
    print(f'Trying to reach: {os.environ.get("VLLM_BASE_URL", "http://localhost:8000/v1")}')

## Step 3: Initialize Embedding Model & ChromaDB

In [ ]:
from src.embeddings import EmbeddingManager

# This will download bge-large-en-v1.5 (~1.3GB) on first run
# Set device='cpu' if GPU should be reserved for vLLM only
emb_manager = EmbeddingManager(
    device='cuda'  # Change to 'cpu' if GPU memory is tight
)

print(f'\nExisting collections: {emb_manager.list_collections()}')

## Step 4: Index Compliance Documents (One-time Setup)

This parses the 3 SEBI regulation PDFs, chunks them by regulation number, 
and indexes them into ChromaDB. Only needs to run once — results persist in `shared/chroma_db/`.

In [ ]:
from src.agent import AuditAgent

# Initialize the agent
agent = AuditAgent(llm_client=llm, embedding_manager=emb_manager)

# Index compliance documents (skip if already done)
compliance_chunks = agent.setup_compliance_rag(force_reindex=False)

## Step 5: Load Compliance Rules

We ship **103 built-in SEBI compliance rules** covering:
- **ICDR Schedule VI** — Cover page, risk factors, capital structure, objects, basis for price, tax, industry, business, management, promoters, dividends, financials, legal info
- **LODR Corporate Governance** — Board composition, committees, whistle blower, insider trading
- **SAST** — Public shareholding, promoter holding

These are ready to use instantly. You can also auto-generate additional rules from the SEBI PDFs using the LLM (takes ~10-15 min).

In [ ]:
# Load built-in 103 rules (instant)
from src.rule_generator import DEFAULT_SEBI_RULES, print_rules_summary

rules = DEFAULT_SEBI_RULES
print(f'Loaded {len(rules)} built-in compliance rules')

# Count by type and severity
by_type = {}
by_sev = {}
for r in rules:
    by_type[r['check_type']] = by_type.get(r['check_type'], 0) + 1
    by_sev[r['severity']] = by_sev.get(r['severity'], 0) + 1
print(f'  By type:     {by_type}')
print(f'  By severity: {by_sev}')

In [ ]:
# View all rules (optional — prints a long list)
# print_rules_summary(rules)

## Step 6: Test Document Parsing

Let's test the document parser on a sample DRHP before running the full audit.

In [ ]:
from src.document_parser import parse_document
from src.config import IPO_DIR

# Pick a sample DRHP to test
sample_files = sorted([f for f in os.listdir(IPO_DIR) if f.endswith('.pdf')])[:5]
print('Available sample DRHPs:')
for i, f in enumerate(sample_files):
    size_mb = os.path.getsize(os.path.join(IPO_DIR, f)) / (1024*1024)
    print(f'  {i}: {f} ({size_mb:.1f} MB)')

# Choose the first one (change index as needed)
SAMPLE_IDX = 0
sample_pdf = os.path.join(IPO_DIR, sample_files[SAMPLE_IDX])
print(f'\nSelected: {sample_files[SAMPLE_IDX]}')

In [ ]:
# Parse the sample document
parsed = parse_document(sample_pdf, verbose=True)

# Show detected sections
print(f'\nDetected Sections ({len(parsed.sections)}):')
for s in parsed.sections:
    print(f'  Pages {s.start_page}-{s.end_page}: {s.name} -> "{s.title[:60]}"')

# Show metadata
print(f'\nMetadata: {parsed.metadata}')

# Show sample tables
all_tables = [t for p in parsed.pages for t in p.tables]
print(f'\nTotal tables found: {len(all_tables)}')
if all_tables:
    print(f'\nFirst table (page {all_tables[0].page_num}):')
    print(all_tables[0].markdown[:500])

## Step 7: Run Full Audit

This runs the complete pipeline:
1. Parse document
2. Index into ChromaDB
3. Run all 103 validation checks (deterministic + semantic + cross-reference)
4. Critic reviews findings
5. Compute confidence scores
6. Generate report

**Expected time: 15-30 minutes** with 103 rules (80 semantic LLM calls + critic review).

In [ ]:
# Run the full audit pipeline
audit_state = agent.run_audit(
    document_path=sample_pdf,
    rules=rules
)

## Step 8: View Results

In [ ]:
# Display findings table
from src.report_generator import print_findings_table

print_findings_table(audit_state.verified_findings)

In [ ]:
# View detailed findings for non-compliant items
import json

non_compliant = [f for f in audit_state.verified_findings if f['status'] != 'COMPLIANT']

print(f'\n{"="*70}')
print(f'  NON-COMPLIANT AND NEEDS-REVIEW FINDINGS ({len(non_compliant)})')
print(f'{"="*70}\n')

for i, finding in enumerate(non_compliant, 1):
    status_icon = {'NON_COMPLIANT': 'FAIL', 'NEEDS_REVIEW': 'REVIEW'}.get(finding['status'], '?')
    print(f'--- Finding {i}: [{status_icon}] {finding["rule_title"]} ---')
    print(f'  Rule ID:     {finding["rule_id"]}')
    print(f'  Regulation:  {finding["regulation_ref"]}')
    print(f'  Severity:    {finding["severity"]}')
    print(f'  Confidence:  {finding["confidence"]:.0%}')
    print(f'  Check Type:  {finding["check_type"]}')
    print(f'  Explanation: {finding.get("explanation", "N/A")}')
    print(f'  Evidence:    {finding.get("evidence", {}).get("document_excerpt", "N/A")[:200]}')
    if finding.get('critic_reasoning'):
        print(f'  Critic Note: {finding["critic_reasoning"]}')
    print()

In [ ]:
# View false positives caught by critic
if audit_state.discarded_findings:
    print(f'\nFalse Positives Caught by Critic Agent ({len(audit_state.discarded_findings)}):')
    print(f'{"-"*60}')
    for d in audit_state.discarded_findings:
        print(f'  Rule: {d["rule_id"]} - {d["rule_title"]}')
        print(f'  Reason: {d.get("discard_reason", "N/A")}')
        print()
else:
    print('\nNo false positives were caught by the critic agent.')

In [ ]:
# View score summary
print(f'\n{"="*50}')
print(f'  COMPLIANCE SCORE SUMMARY')
print(f'{"="*50}')
summary = audit_state.score_summary
print(f'  Overall Score:       {audit_state.overall_score:.0%}')
print(f'  Total Rules Checked: {summary.get("total_rules_checked", 0)}')
print(f'  Compliant:           {summary.get("compliant", 0)}')
print(f'  Non-Compliant:       {summary.get("non_compliant", 0)}')
print(f'  Needs Review:        {summary.get("needs_review", 0)}')
print(f'  Avg Confidence:      {summary.get("average_confidence", 0):.0%}')
print(f'\n  By Severity: {json.dumps(summary.get("by_severity", {}), indent=4)}')
print(f'  By Check Type: {summary.get("by_check_type", {})}')

## Step 9: View Audit Trail

In [ ]:
# Display visual audit trail
print(f'\n{"="*70}')
print(f'  AUDIT TRAIL ({len(audit_state.audit_trail)} steps)')
print(f'{"="*70}\n')

for step in audit_state.audit_trail:
    status = step.get('status', 'unknown')
    icon = {'completed': 'done', 'in_progress': '...', 'error': 'ERR'}.get(status, '?')
    node = step.get('node', 'unknown')
    action = step.get('action', '')
    result = step.get('result', '')
    ts = step.get('timestamp', '')[:19]
    
    if status == 'completed' and result:
        print(f'  [{ts}] [{icon:>4}] {node:<20} {action}')
        print(f'                          -> {result}')
    elif status == 'in_progress':
        print(f'  [{ts}] [{icon:>4}] {node:<20} {action}')

## Step 10: Generate PDF Report

In [ ]:
from src.report_generator import generate_pdf_report
from src.agent import save_report

# Save JSON report
reports_dir = os.path.join(os.environ['AUDIT_BASE_DIR'], 'reports')
json_path = save_report(audit_state, output_dir=reports_dir)

# Generate PDF report
pdf_path = json_path.replace('.json', '.pdf')
generate_pdf_report(audit_state.report, pdf_path)

print(f'\nReports saved to: {reports_dir}')
print(f'  JSON: {json_path}')
print(f'  PDF:  {pdf_path}')

## Step 11: Run on a Different Document

Change the path below to audit any other DRHP from the dataset.

In [ ]:
# Pick another DRHP to audit
# List available documents
ipo_files = sorted([f for f in os.listdir(IPO_DIR) if f.endswith('.pdf')])

print(f'Available documents ({len(ipo_files)}):')
for i, f in enumerate(ipo_files[:20]):
    size_mb = os.path.getsize(os.path.join(IPO_DIR, f)) / (1024*1024)
    print(f'  {i:3d}: {f} ({size_mb:.1f} MB)')
print('  ...')

In [ ]:
# Run audit on a different document
# Change DOCUMENT_INDEX to pick a different file
DOCUMENT_INDEX = 3  # Change this

next_pdf = os.path.join(IPO_DIR, ipo_files[DOCUMENT_INDEX])
print(f'Auditing: {ipo_files[DOCUMENT_INDEX]}')

state2 = agent.run_audit(document_path=next_pdf, rules=rules)

# Show results
print_findings_table(state2.verified_findings)

# Save report
save_report(state2, output_dir=reports_dir)

## Step 12: LLM Usage Statistics

In [ ]:
# Print usage stats
stats = llm.get_stats()
print(f'\nLLM Usage Statistics:')
print(f'  Total API calls: {stats["total_calls"]}')
print(f'  Total tokens:    {stats["total_tokens"]:,}')

critic_stats = agent.critic.get_stats()
print(f'\nCritic Statistics:')
print(f'  Valid findings:     {critic_stats["valid"]}')
print(f'  False positives:    {critic_stats["false_positive"]}')
print(f'  Needs more evidence: {critic_stats["needs_more"]}')
print(f'  Errors:             {critic_stats["errors"]}')

---

## Summary

This notebook demonstrates the full AI-Driven Audit & Compliance Validator:

| Feature | Description |
|---|---|
| **103 Built-in Rules** | Comprehensive SEBI ICDR/LODR/SAST compliance rules |
| **5-Layer Validation** | Deterministic + NLP + Semantic + Numerical + Cross-Reference |
| **Self-Reflection** | Critic agent catches false positives |
| **Confidence Scoring** | Multi-signal weighted formula (not arbitrary numbers) |
| **Full Auditability** | Every step logged with timestamps |
| **Cross-Reference** | Catches numerical mismatches between text and tables |

### Next Steps
- Run the Streamlit dashboard: `streamlit run app.py`
- Build the evaluation pipeline with ground truth labels
- Record the demo video